---

## 研究范围界定：时间与空间

### 一、各数据源时间覆盖汇总

| 数据类别 | 数据集 | 最早可用 | 最晚可用 | 频率 | 约束等级 |
|----------|--------|----------|----------|------|----------|
| **1A 目标变量** | EIA Brent Spot Price | 1987-05 | 2026-05 | Daily | — |
| **1B 基准对比** | EIA WTI Spot Price | 1986-01 | 2026-05 | Daily | — |
| **1C EIA 周报** | Commercial Crude Stocks | 1982 | 2026-05 | Weekly | — |
| | Cushing Crude Stocks | 2004 | 2026-05 | Weekly | — |
| | Crude Production | ~1990s | 2026-05 | Weekly | — |
| | Crude Imports / Exports | ~1990s | 2026-05 | Weekly | — |
| | Refinery Inputs / Utilisation | 1982 / ~1990s | 2026-05 | Weekly | — |
| | Gasoline / Distillate / Jet Fuel Supplied | ~1990s | 2026-05 | Weekly | — |
| **1D 宏观金融** | Yahoo Finance S&P 500 (^GSPC) | 2006-01-03 | 2025-12-31 | Daily | — |
| | FRED Dollar Index (DTWEXBGS) | 2006-01-02 | 2026-05-15 | Daily | — |
| | FRED VIX | 1990-01-02 | 2026-05-21 | Daily | — |
| | FRED 10Y Treasury Yield | 1962-01-02 | 2026-05-21 | Daily | — |
| | FRED Fed Funds Rate | 1954-07-01 | 2026-05-21 | Daily | — |
| **2 官方报告** | OPEC MOMR | 2001 | present | Monthly | — |
| | EIA STEO | — | present | Monthly | — |
| **3 新闻/事件** | GDELT 2.0 | **2015-02** | present | 15-min | ⚠️ |
| | Aramco News (已爬取) | 2025-06 | 2026-05 | Irregular | ⚠️ 极短 |
| | Shell News | 2013 | present | Irregular | — |
| **4 遥感** | Sentinel-2 L2A | **2017-04** | 2025-12 | 5-day → monthly | ⚠️ 瓶颈 |
| | Landsat Collection 2 | 2005 | present | 16-day → monthly | — |
| | VIIRS Night Lights | 2012 | present | Daily/Monthly | — |
| **5 航运** | EMODnet Vessel Density | **2017** | **2024** | Monthly | ⚠️ 瓶颈 |
| | NOAA AIS (US) | 2009 | 2025 | Daily | — |
| | IMF PortWatch | **2019** | present | Daily | ⚠️ |
| | GFW AIS Presence | 2012 | near real-time | Daily | — |
| | GFW SAR Detections | 2017 | present | Satellite pass | — |
| **6 基础设施** | WPI / OGIM / GOGET / GOIT | Static | Static | — | — |
| **7 气象** | ERA5 | 1940 | present | Hourly/Daily | — |
| | IBTrACS | 1842 | present | 6-hourly | — |

#### 2A. 三个数据集在研究中的角色定位

| 维度 | OPEC MOMR | EIA STEO | OPEC ASB |
|---|---|---|---|
| **在消融实验中的位置** | M2（Text 模态） | M2（Text 模态） | 不进入 M1–M4 特征矩阵 |
| **核心角色** | 全球供需叙事的**月度情感 / 事件特征**来源 | 美国视角的**月度市场展望情感**来源 | 供应链结构的**静态背景知识** |
| **处理方式** | PDF → LLM 提取 → 结构化月度特征 → 前向填充至周度 | PDF → LLM 提取 → 结构化月度特征 → 前向填充至周度 | PDF / 表格 → 手动或半自动提取 → 年度参考数据 |
| **输出特征示例** | `momr_supply_sentiment`<br>`momr_demand_revision_direction`<br>`momr_disruption_risk_flag`<br>`momr_production_cut_signal` | `steo_brent_outlook_sentiment`<br>`steo_supply_disruption_flag`<br>`steo_demand_revision_direction` | 不输出时序特征 |
| **时间覆盖** | 2001–present，可覆盖研究期 2006–2025 | Monthly，可覆盖研究期 2006–2025 | 2001–2024 editions |
| **频率对齐** | 月度 → 前向填充至周度；每月报告发布日起生效，直到下一期报告发布 | 月度 → 前向填充至周度 | N/A |

---

#### 具体用途说明

##### ① OPEC MOMR：M2 Text 模态的核心信号源

OPEC Monthly Oil Market Report, MOMR 是全球石油市场最权威的月度供需评估报告之一，包含 OPEC 对全球原油供给、需求、库存、炼厂运行和市场展望的官方判断。

在本研究中，MOMR 主要用于构建 Text 模态中的月度叙事特征：

- 使用 LLM 从每月 PDF 全文中提取**结构化情感 / 事件特征**，例如：
  - 供给侧是否出现意外中断；
  - 需求预期是否上调或下调；
  - OPEC 是否暗示减产或增产；
  - 外部扰动风险是否升级；
  - 库存、炼厂运行或市场平衡是否出现明显变化。

- 提取后的月度特征从报告发布日起**前向填充至周度**，直到下一期报告发布。

- MOMR 提供的是**全球视角**的供需判断，与 EIA Weekly Petroleum Status Report 等美国基本面数据形成互补。

- 在 M2 消融实验中，MOMR 特征与 GDELT、企业新闻等文本数据一起构成 Text 模态的增量贡献。

---

##### ② EIA STEO：M2 Text 模态的补充信号源

EIA Short-Term Energy Outlook, STEO 是美国能源信息署发布的月度市场展望报告，侧重美国和全球液体燃料市场预测，包括原油价格、供需变化、库存、产量和消费趋势等内容。

在本研究中，STEO 主要作为 MOMR 的补充文本信号源：

- 使用 LLM 从 STEO 叙事部分提取与油价预测相关的**结构化文本特征**，例如：
  - Brent 价格展望情感；
  - 供给中断风险；
  - 需求修正方向；
  - 库存压力判断；
  - 市场平衡预期。

- 与 MOMR 的区别在于，STEO 代表的是**美国官方视角**，而 MOMR 更接近 OPEC / 生产国视角。两者之间的叙事差异或情感分歧本身可能成为有价值的预测信号。

- STEO 的表格数据，例如 `EIA_steo_monthly_2022_2027.xlsx`，目前仅保留作参考，**不直接进入特征矩阵**。

- 不直接使用 STEO 数值预测的原因是：STEO 中包含对未来油价和供需的预测值，如果直接作为模型输入，可能引入**信息泄露风险**。因此，本研究只使用其叙事文本中可解释的情感和事件特征。

---

##### ③ OPEC ASB：不进入时序特征矩阵，作为背景参考

OPEC Annual Statistical Bulletin, ASB 是年度统计公报，提供全球石油行业的长期结构性数据，包括 OPEC 和非 OPEC 国家产量、出口量、炼化能力、储量、贸易流向等信息。

在本研究中，ASB **不直接用于周度油价预测模型**，而是作为背景性和解释性数据使用。

其主要用途包括：

- **论文写作的背景数据**  
  用于描述研究期内全球石油供应链的结构变化，例如：
  - OPEC 产量占比变化；
  - 主要产油国出口格局变化；
  - 炼化能力扩张；
  - 全球石油贸易流向转移。

- **AOI 选址论证**  
  引用 ASB 中的国别产量、出口量和炼化能力数据，支撑 8 个 AOI 站点选择的合理性。

- **供应链图构建参考**  
  在 Section 6 的 supply-chain graph 构建中，ASB 可为 OGIM、GOGET、GOIT 等基础设施数据提供产量权重、贸易流向和区域重要性的定量依据。

- **不进入时序建模的原因**  
  ASB 是年度频率，远低于本研究的周度建模频率；同时，其发布通常存在约 1 年滞后。因此，如果直接作为周度模型特征，时间分辨率不足，且可能无法及时反映短期市场变化。

---

#### 小结

| Dataset | 是否进入模型 | 所属模态 | 主要作用 |
|---|---|---|---|
| **OPEC MOMR** | 是 | M2 Text | 全球供需叙事、OPEC 视角、月度情感和事件特征 |
| **EIA STEO** | 是 | M2 Text | 美国官方市场展望、Brent outlook、供需修正信号 |
| **OPEC ASB** | 否 | 背景参考 | 年度结构性数据、AOI 选择依据、供应链图权重参考 |

对比总结表：一目了然地展示三者在消融实验中的位置、核心角色、处理方式、输出特征示例、时间对齐方式的差异。

三个数据集的具体用途：

OPEC MOMR → M2 Text 模态的核心信号源。通过 LLM 从月度 PDF 提取供需情感/事件特征，前向填充至周度，提供全球供需视角。

EIA STEO → M2 Text 模态的补充信号源。同样用 LLM 提取情感特征，但代表美国官方视角（vs OPEC 生产国视角）。特别指出 STEO 的表格数据不直接入模，仅使用叙事文本，以避免信息泄露。

OPEC ASB → 不进入 M1–M4 的时序特征矩阵。仅作为论文写作背景数据、AOI 选址论证依据、供应链图构建参考。年度频率不适合周度建模。

### 二、研究时间范围：统一 20 年周期（2006-01 ~ 2025-12）

#### 核心原则：以市场数据定主轴，遥感/航运作为渐进式增强

统一研究期 **2006-01-01 ~ 2025-12-31**（~20 年），不同模态数据按实际可用性 **分段接入**：

```
2006 ──────────── 2012 ── 2015 ── 2017 ──────── 2025
 │                  │       │       │              │
 ├─ Market+Macro ───┼───────┼───────┼──────────────┤  全程可用
 ├─ OPEC MOMR ──────┼───────┼───────┼──────────────┤  全程可用 (2001+)
 │                  │       │       │              │
 │                  ├─ VIIRS Night Lights ─────────┤  2012+
 │                  │       ├─ GDELT 2.0 ──────────┤  2015-02+
 │                  │       │       │              │
 ├─ Landsat ────────┼───────┼───────┼──────────────┤  2005+ (30m, 16-day)
 │                  │       │       ├─ Sentinel-2 ─┤  2017-04+ (10m, 5-day)
 │                  │       │       ├─ EMODnet ────┤  2017–2024
 │                  │       │       ├─ GFW SAR ────┤  2017+
 ├─ NOAA AIS (US) ──┼───────┼───────┼──────────────┤  2009+
 │                  ├─ GFW AIS ─────┼──────────────┤  2012+
 │                  │       │       │  ├ PortWatch ┤  2019+
 │                  │       │       │              │
2006 ──────────── 2012 ── 2015 ── 2017 ──────── 2025
```

#### 解决遥感与航运数据时间缺口的三种策略

**策略 A：用 Landsat 回填遥感至 2006（推荐）**

| 时段 | 遥感数据源 | 分辨率 | 重访周期 | 可提取指标 |
|------|-----------|--------|----------|-----------|
| 2006-01 ~ 2017-03 | **Landsat 5/7/8** Collection 2 L2 | 30 m | 16 天 → 月度合成 | NDVI, NDWI, NDBI, BSI |
| 2017-04 ~ 2025-12 | **Sentinel-2** L2A + Landsat 8/9 | 10 m (S2) / 30 m (LS) | 5 天 (S2) → 月度合成 | NDVI, NDWI, NDBI, BSI |

> Landsat 和 Sentinel-2 提取的是**相同的光谱指标**（NDVI/NDWI/NDBI/BSI），只是空间分辨率不同（30m vs 10m）。由于最终聚合到 AOI 级别的月度均值/标准差，两者在特征空间上是**连续可比的**。可加入 `sensor_flag` 控制变量标记数据来源。

**策略 B：航运特征渐进可用 + 缺失标记**

| 时段 | 可用航运数据 | 处理方式 |
|------|-------------|---------|
| 2006-01 ~ 2008-12 | 无 | 航运特征设为 NaN，加入 `shipping_available = 0` 指示变量 |
| 2009-01 ~ 2011-12 | NOAA AIS (US 沿海) | 仅美国港口航运特征可用 |
| 2012-01 ~ 2016-12 | + GFW AIS (全球) | 全球油轮存在时长可用 |
| 2017-01 ~ 2018-12 | + EMODnet + GFW SAR | 欧洲航线密度 + 全球 SAR 船舶检测 |
| 2019-01 ~ 2025-12 | + IMF PortWatch | 完整咽喉通行量（最佳覆盖阶段） |

> 使用支持缺失值的模型（如 XGBoost、LightGBM 原生支持 NaN；Transformer 可用 mask），或构建 `modality_availability` 二进制向量，让模型学习在不同数据可用条件下做预测。

**策略 C：消融实验设计（验证每种模态的增量贡献）**

| 模型 | 特征集 | 数据可用期 | 目的 |
|------|--------|-----------|------|
| M1 | Market + Macro | 2006–2025 全程 | 纯基本面基准 |
| M2 | M1 + Text (OPEC, GDELT, News) | 2006–2025（GDELT 2015+） | + NLP 增量 |
| M3 | M2 + Remote Sensing (Landsat→S2) | 2006–2025 全程 | + 遥感增量 |
| M4 | M3 + Shipping (渐进接入) | 2006–2025（渐进） | + 航运增量 = Full Multimodal |
| M4* | M4 在 2017–2025 子集上 | 2017–2025 | 密集数据期对比（所有模态完整可用） |

> 通过 M1→M4 的递进对比，可以清晰展示**每种模态的边际贡献**，这对论文叙事非常有力。

### 三、研究地理范围

#### 3.1 AOI 站点：5 → 11 个

**选站框架：throughput/capacity rank × geographic/supply-chain diversity × remote sensing observability**

按供应链角色分为三类：出口终端（Supply）、中转仓储（Transit）、进口/炼化（Demand），确保 OPEC 前三大产油国出口设施全覆盖 + 五大消费区各一个最大设施。

**11 个 AOI：**

| Site ID | 站点名称 | 类型 | 国家 | 区域 | 坐标 (lon, lat) | 产能/吞吐量 | 全球排名 | 选取理由 |
|---------|---------|------|------|------|-----------------|------------|---------|---------|
| P001 | Port of Rotterdam | 港口 | 荷兰 | 欧洲 | 4.145, 51.950 | 397M tonnes/yr (2024) | 欧洲 #1 港口 | Brent 定价体系实物枢纽；欧洲需求代理 |
| P002 | Fujairah Oil Terminal | 油库 | 阿联酋 | 中东 | 56.336, 25.128 | 1.5M bpd ADCOP bypass | 全球 #2 加油枢纽 | 霍尔木兹 bypass 战略节点 |
| P003 | Ras Tanura Terminal | 出口终端 | 沙特 | 中东 | 50.157, 26.643 | 9M bpd 设计容量 | 全球 #1 出口终端 | 沙特 90% 碳氢出口经此 |
| P004 | Singapore Jurong Island | 炼厂 | 新加坡 | 东南亚 | 103.708, 1.269 | 605,000 bpd | 全球 #12 炼厂 | 马六甲海峡核心炼化节点 |
| P005 | Houston Ship Channel | 港口 | 美国 | 北美 | -95.085, 29.732 | >3M bpd 炼厂集群 | 美国 #1 水运港 | 北美需求代理；Gulf Coast hub |
| P006 | Ningbo-Zhoushan Port | 进口港 | 中国 | 东亚 | 121.93, 29.87 | 185M tonnes/yr 原油 | 全球 #1 货物吞吐量 | 中国需求代理 |
| P007 | Jamnagar Refinery | 炼厂 | 印度 | 南亚 | 69.67, 22.47 | 1,240,000 bpd | 全球 #1 炼厂 | 印度需求代理 |
| P008 | Basra Oil Terminal | 出口终端 | 伊拉克 | 中东 | 48.80, 29.69 | >3.3M bpd | 中东 #3 出口终端 | 伊拉克 95%+ 出口；OPEC #2 产油国 |
| **P009** | **Ulsan Refinery** | **炼厂** | **韩国** | **东亚** | **129.343, 35.433** | **840,000 bpd** | **全球 #3 炼厂** | **东亚炼化代理；填补韩国缺口（3 座 Top 10 炼厂）** |
| **P010** | **Kharg Island Terminal** | **出口终端** | **伊朗** | **中东** | **50.310, 29.245** | **1.5–2.5M bpd** | **伊朗唯一主要出口枢纽** | **OPEC #3 产油国；90–96% 伊朗原油出口经此** |
| **P011** | **Yanbu Export Terminal** | **出口终端** | **沙特** | **中东** | **38.05, 24.08** | **4.5M bpd 名义容量** | **红海 #1 原油终端** | **Petroline 管道终点；Hormuz bypass 替代路线** |

> **覆盖：8 个国家、5 个区域**（中东 5 站、东亚 2 站、东南亚 1 站、南亚 1 站、欧洲 1 站、北美 1 站）。供应链全链条：出口终端 4 站 + 中转仓储 2 站 + 进口/炼化 5 站。
>
> **数据来源**：炼厂排名引自 Oil & Gas Journal（Wikipedia "List of oil refineries"）；终端吞吐量引自 IMF PortWatch、Saudi Aramco、Marine Insight；港口排名引自 Eurostat (2025)、新华社。OGIM 数据库确认全部 11 个设施 OPERATIONAL 状态。
>
> **文献先例**：Wang et al. (2023, *Nature HSS Comms*) 选 8 个美国储油区（PADD2+3 占 US 库存 70%+）；Guetta-Jeanrenaud et al. (2025, arXiv) 选 64 个美国港口（按 WPI 规模分级）。
>
> **稳健性**：配合 leave-one-AOI-out 敏感性测试，验证模型结果不依赖于任何单一站点的纳入或排除。

#### 3.2 海运咽喉要道：6 个足够

| 咽喉要道 | 区域 | 石油通行量 | 状态 |
|----------|------|-----------|------|
| Strait of Hormuz | 波斯湾 → 印度洋 | ~21 mb/d (全球 ~21%) | ✓ 核心 |
| Suez Canal | 地中海 ↔ 红海 | ~9 mb/d | ✓ 核心 |
| Strait of Malacca | 印度洋 → 南海 | ~16 mb/d | ✓ 核心 |
| Bab el-Mandeb | 红海 → 亚丁湾 | ~9 mb/d | ✓ 核心 |
| Cape of Good Hope | 非洲南端 | ~9 mb/d (替代路线) | ✓ 核心 |
| Panama Canal | 大西洋 ↔ 太平洋 | ~1 mb/d | ✓ 补充 |

> **6 个足够的理由：**
> 1. 这 6 个是 **EIA 官方定义的全球石油海运咽喉要道**，已是学术界和行业的标准集合。
> 2. IMF PortWatch 恰好覆盖这 6 个，数据获取零额外成本。
> 3. 其他候选（如 Turkish Straits/博斯普鲁斯、Danish Straits/丹麦海峡）的石油通行量远小于上述 6 个，信息增益有限。
> 4. Houthi 危机（2023-24）导致的红海航线绕行 Cape of Good Hope 事件，正好可以通过 Bab el-Mandeb + Suez + Cape 三者的流量变化来捕捉——当前 6 个已完整覆盖此类事件。

#### 3.3 市场基本面覆盖

| 空间维度 | 数据源 | 说明 |
|----------|--------|------|
| 美国（供需基本面） | EIA 周报全系列 | 库存、产量、进出口、炼厂运行、成品油消费——全球最透明的石油市场统计 |
| OPEC（供给侧） | OPEC MOMR | 全球供需叙事、OPEC 产量决策 |
| 全球（扰动事件） | GDELT 2.0 | 冲突、制裁、运输中断事件信号 |
| 全球（气象灾害） | ERA5 + IBTrACS | 飓风、极端天气对炼厂/航运的干扰 |

### 四、总结

#### 研究时间范围

| | 起止时间 | 长度 | 说明 |
|--|---------|------|------|
| **统一研究期** | **2006-01-01 ~ 2025-12-31** | **~20 年** | 以市场数据为主轴 |
| 遥感覆盖 | Landsat 2006+ → Sentinel-2 2017+ | 全程 | Landsat 回填 2006-2016 |
| 航运覆盖 | 渐进接入：NOAA 2009+ / GFW 2012+ / PortWatch 2019+ | 渐进 | 缺失期用 NaN + 指示变量 |
| 密集数据子期 | 2017-04 ~ 2025-12 | ~8.7 年 | 所有模态完整可用，用于 M4* 对比实验 |

#### 研究空间范围

| 维度 | 范围 | 数量 |
|------|------|------|
| **遥感 AOI** | Rotterdam, Fujairah, Ras Tanura, Singapore, Houston, Ningbo-Zhoushan, Jamnagar, Basra, **Ulsan, Kharg Island, Yanbu** | **11 个** |
| **航运咽喉** | Hormuz, Suez, Malacca, Bab el-Mandeb, Cape of Good Hope, Panama | **6 个** |
| **市场基本面** | 美国 EIA 周报 + OPEC 全球供需报告 | 全球 |
| **扰动事件/气象** | GDELT + ERA5 + IBTrACS | 全球 |

#### 待办事项
- [x] ~~重新从 FRED 下载 S&P 500 数据至 2006-01-01~~ → 已改用 Yahoo Finance 下载，覆盖 2006-01-03 ~ 2025-12-31
- [x] ~~为 3+3 个新增 AOI (P006–P011) 配置 GEE 提取脚本~~ → 已完成，11 站全量导出
- [x] ~~编写 Landsat Collection 2 回填脚本（2006-2016，与 Sentinel-2 指标对齐）~~ → 已完成
- [ ] 决定 Aramco / Shell News 爬取范围是否需要回溯至更早年份
- [ ] 确认 EMODnet 2024H2 数据质量后决定航运特征截止处理方式
- [ ] 构建 `modality_availability` 指示向量，用于模型训练中标记各时段特征可用性

**7) Weather & Auxiliary**

| Dataset | Summary | URL | Variables | Time Range | Frequency | Coverage | Data Type | Access | Potential Use | Limitations |
|---------|---------|-----|-----------|------------|-----------|----------|-----------|--------|---------------|-------------|
| ERA5 Daily Statistics (Copernicus) | Global reanalysis weather data | [CDS](https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics) | Wind, waves, temperature, precipitation, pressure | 1940 -- present | Hourly / Daily | Global (0.25 deg grid) | Weather / reanalysis | Open (CDS registration) | Shipping condition controls, refinery disruption risk, extreme weather flags | Large data volume; requires spatial subsetting; CDS registration needed |
| NOAA IBTrACS | Global tropical cyclone best-track archive | [NOAA](https://www.ncei.noaa.gov/products/international-best-track-archive) | Storm track, intensity, wind speed, pressure, landfall | 1842 -- present | 6-hourly (per storm) | Global (all ocean basins) | Weather / event data | Open | Hurricane/typhoon event dummies, path-based disruption impact variables | Sparse events; not all storms affect oil infrastructure; requires spatial matching |